# D'QUILANE Global v0.4 — 30+ Languages + GitHub Pages

Multilingual LLM (balita level) yang bisa generate text dari **30+ bahasa** dengan **native script**:
- Latin (Indo, English, Spanish, French, German, Italian, Portuguese, Polish, Czech, Swedish, Romanian, Tagalog, Catalan, Finnish, Norwegian, Dutch, Turkish, Vietnamese)
- Cyrillic (Russian, Ukrainian)
- CJK (Chinese, Japanese with Kanji)
- Hiragana + Katakana (Japanese)
- Hangul (Korean)
- Arabic (Arabic)
- Devanagari (Hindi)
- Bengali, Thai, Greek, Hebrew

## Workflow:
1. Run training (cell 1-5)
2. Export JSON (cell 6) — file untuk GitHub Pages
3. Download `dquilane_model.json` dari tab Output
4. Push ke GitHub repo + aktifkan Pages
5. Web tester live di `https://<username>.github.io/<repo>/`

In [ ]:
# ===== CELL 1: INSTALL =====
import subprocess, sys
for pkg in ['torch', 'gradio']:
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
import torch, os, math, json, time
import torch.nn as nn
import torch.nn.functional as F
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}  Device: {DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ===== CELL 2: MULTILINGUAL CORPUS (30+ languages, native script) =====
# Edit/tambah kalimat baru di sini. Tiap bahasa 9-10 kalimat kindergarten level.
TEXT = '''Indonesia adalah negara kepulauan.
Ibu kota Indonesia adalah Jakarta.
Bahasa Indonesia adalah bahasa resmi.
Penduduk Indonesia sangat beragam.
Indonesia kaya akan sumber daya alam.
Mari belajar bahasa Indonesia.
Anak muda harapan masa depan.
Belajar setiap hari bikin hidup bermakna.
Kerja keras kunci pencapaian tujuan.
Indonesia akan terus maju dan berkembang.
English is a global language.
The world is connected by technology.
Computers changed how humans communicate.
Artificial intelligence is a global trend.
Large language models are popular now.
Children learn languages quickly.
Education is the key to the future.
Hard work leads to success.
Knowledge is power and wisdom.
Every day is a chance to learn.
El espanol es un idioma hermoso.
La familia es muy importante.
El mundo es grande y diverso.
La educacion es la clave del futuro.
El trabajo duro trae exito.
Cada dia es una nueva oportunidad.
Los ninos aprenden rapidamente.
La tecnologia avanza rapidamente.
El conocimiento es poder.
Le francais est une belle langue.
La famille est tres importante.
Le monde est grand et diversifie.
L education est la cle du futur.
Le travail dur mene au succes.
Chaque jour est une chance d apprendre.
Les enfants apprennent vite.
La technologie avance rapidement.
Le savoir est le pouvoir.
Deutsch ist eine wichtige Sprache.
Die Familie ist sehr wichtig.
Die Welt ist gross und vielfaeltig.
Bildung ist der Schluessel zur Zukunft.
Harte Arbeit fuehrt zum Erfolg.
Jeder Tag ist eine neue Chance.
Kinder lernen schnell.
Die Technik schreitet voran.
Wissen ist Macht.
Il italiano e una bella lingua.
La famiglia e molto importante.
Il mondo e grande e diversificato.
L istruzione e la chiave del futuro.
Il lavoro duro porta al successo.
Ogni giorno e una nuova opportunita.
I bambini imparano velocemente.
La tecnologia avanza rapidamente.
La conoscenza e potere.
Portugues e uma lingua bonita.
A familia e muito importante.
O mundo e grande e diversificado.
A educacao e a chave do futuro.
O trabalho duro leva ao sucesso.
Cada dia e uma nova oportunidade.
As criancas aprendem rapidamente.
A tecnologia avanca rapidamente.
O conhecimento e poder.
Русский язык использует кириллицу.
Русским языком говорят многие люди.
Многие страны используют кириллицу.
Языки соединяют разные народы.
Чтение расширяет человеческий разум.
Письмо проясняет наши мысли.
Речь строит уверенность в себе.
Слушание это первый навык.
Знания связывают поколения.
Тяжелая работа приносит хорошие результаты.
中文是世界上使用人数最多的语言。
许多人每天说普通话。
写中文使用汉字。
中国的万里长城很有名。
学习打开新的大门。
教育建设强大的国家。
家庭非常重要。
耐心会带来巨大的回报。
诚实是最好的策略。
时间比金子更有价值。
日本語には三つの書き方があります。
ひらがなは日本語の単語に使います。
カタカナは外国語の言葉に使います。
漢字は中国語の文字を表します。
日本は島国です。
東京は首都です。
富士山はとても美しいです。
桜は春に咲きます。
教育は高く評価されています。
他人を尊重することが大切です。
한국어는 한글을 사용합니다.
한글은 과학적인 알파벳입니다.
서울은 한국의 수도입니다.
김치는 전통 음식입니다.
가족은 사회의 중심입니다.
교육은 많은 문을 엽니다.
노력은 성공을 가져옵니다.
정직은 강한 신뢰를 구축합니다.
음악은 모든 사람을 연결합니다.
언어 학습은 가치 있습니다.
العربية تكتب من اليمين إلى اليسار.
الخط العربي جميل جدا.
العديد من اللغات تستخدم الحروف العربية.
الشرق الأوسط غني بالثقافة.
المعرفة تحظى بتقدير كبير.
روابط الأسرة قوية جدا.
الضيافة قيمة أساسية.
الصبر فضيلة عظيمة.
التعليم ينير العقل.
القراءة توسع الآفاق.
हिन्दी देवनागरी लिपि का उपयोग करती है.
हिन्दी लाखों लोग बोलते हैं.
भारत में कई विविध भाषाएं हैं.
नई दिल्ली राजधानी है.
ताजमहल प्रसिद्ध है.
परिवार के मूल्य बहुत मजबूत हैं.
शिक्षा का उच्च मूल्य है.
त्योहार लोगों को एक साथ लाते हैं.
संगीत और नृत्य समुदायों को जोड़ते हैं.
ज्ञान आगे का रास्ता रोशन करता है.
বাংলা একটি সুন্দর ভাষা.
বাংলাদেশ বাংলা ব্যাপকভাবে ব্যবহার করে.
কলকাতাও বাংলা বলে.
বাংলা লিপি বাঁকা.
সাহিত্য খুব সমৃদ্ধ.
ট্যাগর একজন মহান কবি ছিলেন.
সঙ্গীত সংস্কৃতির অংশ.
মাছ একটি সাধারণ খাবার.
পারিবারিক বন্ধন শক্তিশালী.
শিক্ষা জীবন রূপান্তরিত করে.
ภาษาไทยมีห้าเสียงวรรณยุกต์.
กรุงเทพคือเมืองหลวง.
อักษรไทยสวยงาม.
พุทธศาสนาได้รับการปฏิบัติอย่างกว้างขวาง.
ยิ้มแสดงความเคารพที่นี่.
ครอบครัวคือหน่วยหลัก.
อาหารเผ็ดและอร่อย.
เทศกาลเป็นเหตุการณ์ที่มีสีสัน.
การศึกษามีคุณค่าสูง.
เคารพผู้ใหญ่เสมอ.
Türkçe Latin harflerini kullanır.
İstanbul ünlü bir şehirdir.
Türkiye iki kıtayı birleştirir.
Türk kahvesi çok ünlüdür.
Aile çok önemlidir.
Eğitim çok değerlidir.
Misafirperverlik temel bir değerdir.
Çalışmak başarıyı getirir.
Müzik insanları birleştirir.
Dil öğrenmek ödüllendiricidir.
Tiếng Việt sử dụng bảng chữ cái Latin.
Hà Nội là thủ đô.
Việt Nam có lịch sử lâu dài.
Tiếng Việt có sáu thanh điệu.
Gia đình rất quan trọng.
Giáo dục mở ra nhiều cánh cửa.
Cơm là thức ăn chính.
Gỏi cuốn rất ngon.
Tết là năm mới.
Chăm chỉ dẫn đến thành công.
Polski to język słowiański.
Warszawa jest stolicą.
Polska ma bogatą historię.
Polski używa alfabetu łacińskiego.
Rodzina jest bardzo ważna.
Edukacja otwiera wiele drzwi.
Praca przynosi sukces.
Uczciwość buduje silne zaufanie.
Muzyka łączy pokolenia.
Literatura wzbogaca duszę.
Nederlands wordt in Nederland gesproken.
Amsterdam is de hoofdstad.
Tulpen zijn beroemd in Holland.
Molens zijn hier traditioneel.
Gezin is heel belangrijk.
Onderwijs wordt hoog gewaardeerd.
Fietsen worden veel gebruikt.
Kaas is een beroemde export.
Hard werken brengt succes.
Tolerantie is een kernwaarde.
Ελληνικά χρησιμοποιούν το ελληνικό αλφάβητο.
Η Αθήνα είναι η πρωτεύουσα.
Η δημοκρατία ξεκίνησε στην αρχαία Ελλάδα.
Η ελληνική φιλοσοφία είναι παγκοσμίως γνωστή.
Η Ακρόπολη είναι ιστορική.
Η οικογένεια είναι πολύ σημαντική.
Η εκπαίδευση εκτιμάται ιδιαίτερα.
Το ελαιόλαδο χρησιμοποιείται ευρέως.
Η μουσική συνδέει τις γενιές.
Η γλώσσα έχει αρχαίες ρίζες.
עברית נכתבת מימין לשמאל.
ירושלים היא עיר קדושה.
עברית משתמשת באלף בית ייחודי.
לישראל יש הרבה חדשנות.
משפחה חשובה מאוד.
חינוך מוערך מאוד.
היסטוריה עשירה ועמוקה.
כנות הוא ערך מרכזי.
עבודה קשה מביאה הצלחה.
שלום היא המטרה הסופית.
Kiswahili kinazungumzwa Afrika.
Kiswahili hutumia alfabeti ya Kilatini.
Waafrica wengi wanazungumza Kiswahili.
Nairobi ni jji kuu.
Familia ni muhimu sana.
Elimu inafungua milango mingi.
Muziki ni sehemu ya utamaduni.
Usimulazi ni utamaduni.
Kazi nguvu inaleta mafanikio.
Jumuiya ni kitovu.
Українська використовує кирилицю.
Київ є столицею.
Україна має багаті традиції.
Українська мова мелодійна.
Сім'я дуже важлива.
Освіта високо цінується.
Чесність будує сильну довіру.
Тяжка праця приносить успіх.
Музика поєднує покоління.
Література збагачує душу.
'''

In [ ]:
# ===== CELL 3: CONFIG + TOKENIZER =====
CONFIG = {
    'vocab_size': 0,  # auto-set
    'd_model':    128,    # naik dari 96 utk handle vocab besar
    'n_heads':    4,
    'd_ff':       512,
    'n_layers':   8,      # virtual depth (parameter sharing)
    'max_seq':    80,     # context window
    'epochs':     2000,
    'batch_size': 32,
    'lr':         3e-4,
}

# Char-level tokenizer (handles ANY unicode)
chars = sorted(set(TEXT))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
stoi['<pad>'] = len(stoi)
itos[len(itos)] = '<pad>'
PAD_ID = stoi['<pad>']
VOCAB = len(stoi)
CONFIG['vocab_size'] = VOCAB

def encode(s):
    return [stoi[c] for c in s if c in stoi]
def decode(ids):
    return ''.join(itos.get(int(i), '?') for i in ids if i != PAD_ID)

data = torch.tensor(encode(TEXT), dtype=torch.long)
print(f'Vocab size: {VOCAB} karakter multilingual')
print(f'Corpus: {len(TEXT)} chars, {len(data)} tokens')
print(f'Languages: 30+ (Latin, Cyrillic, CJK, Hangul, Arabic, Devanagari, Bengali, Thai, Greek, Hebrew)')

def get_batch():
    idx = torch.randint(0, len(data) - CONFIG['max_seq'] - 1, (CONFIG['batch_size'],))
    x = torch.stack([data[i:i+CONFIG['max_seq']] for i in idx])
    y = torch.stack([data[i+1:i+1+CONFIG['max_seq']] for i in idx])
    return x.to(DEVICE), y.to(DEVICE)

In [ ]:
# ===== CELL 4: MODEL D'QUILANE =====
class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dh)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class FFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.val  = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.val(x))

class Block(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)
        self.attn  = Attention(d, h)
        self.norm2 = nn.RMSNorm(d)
        self.ffn   = FFN(d, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class DQuilane(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.pos_emb = nn.Embedding(cfg['max_seq'], cfg['d_model'])
        self.block = Block(cfg['d_model'], cfg['n_heads'], cfg['d_ff'])
        self.norm_f = nn.RMSNorm(cfg['d_model'])
        self.head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for _ in range(self.cfg['n_layers']):
            x = self.block(x)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg['vocab_size']),
                                    targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new=50, temp=0.8, top_k=10):
        self.eval()
        for _ in range(max_new):
            x = idx if idx.size(1) <= self.cfg['max_seq'] else idx[:, -self.cfg['max_seq']:]
            logits, _ = self(x)
            logits = logits[:, -1, :] / temp
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = DQuilane(CONFIG).to(DEVICE)
n = sum(p.numel() for p in model.parameters())
print(f"D'QUILANE Global v0.4 lahir!")
print(f'  Unique params: {n:,}  (~{n*4/1024:.1f} KB)')
print(f'  Virtual depth: {CONFIG["n_layers"]}')
print(f'  Effective (no share): {n * CONFIG["n_layers"]:,}')
print(f'  Vocab: {VOCAB} chars (30+ languages)')

In [ ]:
# ===== CELL 5: TRAINING =====
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])
model.train()
t0 = time.time()
losses = []
for epoch in range(CONFIG['epochs']):
    xb, yb = get_batch()
    opt.zero_grad()
    _, loss = model(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    losses.append(loss.item())
    if epoch % 200 == 0 or epoch == CONFIG['epochs'] - 1:
        print(f'epoch {epoch+1:4d}/{CONFIG["epochs"]}  loss={loss.item():.4f}  t={time.time()-t0:.1f}s')
print(f'\nDone in {time.time()-t0:.1f}s. Final loss: {loss.item():.4f}')
model.eval()

In [ ]:
# ===== CELL 6: TEST GENERATE (multilingual) =====
test_prompts = [
    'Indonesia', 'English is', 'Русский', '日本語', '한국어',
    'العربية', 'हिन्दी', 'ภาษาไทย', 'Ελληνικά', 'Tiếng Việt'
]
for p in test_prompts:
    ids = torch.tensor([encode(p)], dtype=torch.long, device=DEVICE)
    out = model.generate(ids, max_new=40, temp=0.7, top_k=10)
    text = decode(out[0].tolist())
    print(f'\n[{p}]')
    print(f'  {text}')

In [ ]:
# ===== CELL 7: EXPORT MODEL KE JSON (UTK GITHUB PAGES) =====
import json, os

weights = {
    'config': CONFIG,
    'stoi': stoi,
    'itos': {str(k): v for k, v in itos.items()},
    'weights': {
        'tok_emb_weight': model.tok_emb.weight.detach().cpu().tolist(),
        'pos_emb_weight': model.pos_emb.weight.detach().cpu().tolist(),
        'norm1_weight': model.block.norm1.weight.detach().cpu().tolist(),
        'attn_qkv_weight': model.block.attn.qkv.weight.detach().cpu().tolist(),
        'attn_out_weight': model.block.attn.out.weight.detach().cpu().tolist(),
        'norm2_weight': model.block.norm2.weight.detach().cpu().tolist(),
        'ffn_gate_weight': model.block.ffn.gate.weight.detach().cpu().tolist(),
        'ffn_val_weight': model.block.ffn.val.weight.detach().cpu().tolist(),
        'ffn_down_weight': model.block.ffn.down.weight.detach().cpu().tolist(),
        'norm_f_weight': model.norm_f.weight.detach().cpu().tolist(),
    }
}

os.makedirs('output', exist_ok=True)
JSON_PATH = 'output/dquilane_model.json'
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(weights, f, ensure_ascii=False)

size_kb = os.path.getsize(JSON_PATH) / 1024
print(f'JSON exported: {JSON_PATH} ({size_kb:.1f} KB)')
print(f'\n>>> DOWNLOAD file ini dari tab Output <<<')
print(f'>>> Upload ke folder web/ di GitHub repo kamu <<<')
print(f'>>> Web tester siap pakai! <<<')

In [ ]:
# ===== CELL 8 (OPTIONAL): GRADIO WEB TESTER =====
# Skip kalau mau langsung ke GitHub Pages
import gradio as gr

def generate_gradio(prompt, max_new, temp, top_k):
    ids = encode(prompt)
    if not ids:
        return '[prompt gak ada karakter yang dikenal]'
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    out = model.generate(idx, max_new=int(max_new), temp=float(temp), top_k=int(top_k))
    return decode(out[0].tolist())

with gr.Blocks(title="D'QUILANE Global Tester") as demo:
    gr.Markdown("""
    # D'QUILANE Global v0.4
    Multilingual LLM balita — 30+ bahasa, native script (Latin, Cyrillic, CJK, Arabic, dll).
    """)
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label='Prompt', value='Indonesia', lines=2)
            max_new = gr.Slider(20, 300, value=80, step=10, label='Max tokens')
            temp = gr.Slider(0.1, 2.0, value=0.7, step=0.1, label='Temperature')
            top_k = gr.Slider(0, 50, value=10, step=1, label='Top-k')
            btn = gr.Button('Generate', variant='primary')
        with gr.Column():
            output = gr.Textbox(label='Output', lines=12)
    btn.click(generate_gradio, [prompt, max_new, temp, top_k], output)
    gr.Examples([
        ['Indonesia'], ['English is'], ['Русский'], ['日本語'], ['한국어'],
        ['العربية'], ['हिन्दी'], ['ภาษาไทย'], ['Ελληνικά'], ['Tiếng Việt']
    ], inputs=prompt)

demo.launch(share=True, debug=False)